In [ ]:
import fsspec
import shapely
import numpy as np
import pandas as pd
import geopandas as gpd
import ultraplot as uplt
import pyarrow.compute as pc
import pyarrow.parquet as pq
from pathlib import Path
from urllib.parse import urljoin
from IPython.display import display
pd.set_option('display.max_rows',10)

In [ ]:
CATALOGURL   = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
RAWDIR       = Path('../data/raw')
PROCESSEDDIR = Path('../data/processed')
FIGDIR       = Path('../figs')
PILOT        = 'ee/ee_2024'
COLLECTIONS  = ['ee/ee_2024','sk/sk_2024','lv/lv_2024','cz/cz_2024','be_vlg/be_vlg_2024','dk/dk_2024','es_cat/es_cat_2024','si/si_2024','fi/fi_2024',
                'ie/ie_2024','se/se_2024','lt/lt_2024','hr/hr_2024','nl/nl_2024','es_vc/es_vc_2024','at/at_2024','es_cm/es_cm_2024','fr/fr_2024']
FILECOLUMNS  = {'fields':['id','hcat:code','hcat:name','geometry'],
                'field_embeddings':['id','pixels','pixel_rule','aef_mean','aef_std','olmoearth_mean','olmoearth_std'],
                'chips':['id','split','geometry']}
MAXFIELDS    = 200000
MINFIELDS    = 100
MINPIXELS    = 5
DROPCROPS    = []
SEED         = 42
SPLITCOLORS  = {'train':'#2a78d6','val':'#eb6834','test':'#1baf7a'}
REMOTE       = fsspec.filesystem('https')
for directory in (RAWDIR,PROCESSEDDIR,FIGDIR):
    directory.mkdir(parents=True,exist_ok=True)

In [ ]:
def build_file_url(collection,filetype):
    return urljoin(CATALOGURL,f"{collection}/{collection.split('/')[1]}_{filetype}.parquet")

def build_raw_path(collection,filetype):
    return RAWDIR/f"{collection.split('/')[1]}_{filetype}.parquet"

def save_raw_copy(collection,filetype):
    rawpath = build_raw_path(collection,filetype)
    if not rawpath.exists():
        partpath = rawpath.with_suffix('.part')
        with REMOTE.open(build_file_url(collection,filetype)) as file:
            parquet = pq.ParquetFile(file)
            with pq.ParquetWriter(partpath,parquet.schema_arrow.__class__([parquet.schema_arrow.field(col) for col in FILECOLUMNS[filetype]],
                                                                         metadata=parquet.schema_arrow.metadata),compression='zstd') as writer:
                for batch in parquet.iter_batches(batch_size=100000,columns=FILECOLUMNS[filetype]):
                    writer.write_batch(batch)
        partpath.rename(rawpath)

def drop_fields(table,keep,reason):
    print(f'- dropped {(~keep).sum():,} that {reason}')
    return table[keep]

def save_figure(fig,name):
    fig.savefig(FIGDIR/f'{name}.png',dpi=200)

In [ ]:
def build_table(collection):
    for filetype in FILECOLUMNS:
        save_raw_copy(collection,filetype)

    labels = pd.read_parquet(build_raw_path(collection,'fields'),columns=['id','hcat:name'])
    pixels = pd.read_parquet(build_raw_path(collection,'field_embeddings'),columns=['id','pixels','pixel_rule'])
    fields = labels.merge(pixels.drop_duplicates('id'),on='id',how='left').rename(columns={'hcat:name':'crop'})
    print(f'{collection} starts with {len(fields):,} fields...')
    fields = drop_fields(fields,fields.crop.notna(),'have no crop label')
    fields = drop_fields(fields,~fields.id.duplicated(keep=False),'share their id with another field')
    fields = drop_fields(fields,(fields.pixels>=MINPIXELS)&(fields.pixel_rule!='touched'),f'are smaller than {MINPIXELS} pixels')
    fields = drop_fields(fields,~fields.crop.isin(DROPCROPS),'are in DROPCROPS')
    counts = fields.crop.value_counts()
    fields = drop_fields(fields,fields.crop.map(counts)>=MINFIELDS,f'belong to the {(counts<MINFIELDS).sum()} crops with < {MINFIELDS} fields')
    if len(fields)>MAXFIELDS:
        quota  = (fields.crop.value_counts()*MAXFIELDS/len(fields)).round().clip(lower=MINFIELDS)
        fields = fields.sample(frac=1,random_state=SEED)
        fields = drop_fields(fields,fields.groupby('crop').cumcount()<fields.crop.map(quota),f'were sampled out to cap the table at ~{MAXFIELDS:,} fields')

    embeddings = pq.read_table(build_raw_path(collection,'field_embeddings'),filters=pc.field('id').isin(fields.id),
                               columns=['id','aef_mean','aef_std','olmoearth_mean','olmoearth_std'])
    names      = [f'{model}{i:03d}' for model in ('aef','olmoearth') for i in range(128)]
    values     = np.hstack([np.stack(embeddings[col].to_numpy(zero_copy_only=False)) for col in ('aef_mean','aef_std','olmoearth_mean','olmoearth_std')])
    features   = pd.DataFrame(values.astype('float32'),columns=names).assign(id=embeddings['id'].to_pylist())

    geoms     = gpd.read_parquet(build_raw_path(collection,'fields'),columns=['id','geometry'],filters=pc.field('id').isin(fields.id))
    geoms     = geoms.assign(geometry=geoms.make_valid())
    chips     = gpd.read_parquet(build_raw_path(collection,'chips')).rename(columns={'id':'chip'})
    pairs     = gpd.sjoin(geoms,chips,predicate='intersects')
    pairs     = pairs.assign(overlap=shapely.area(shapely.intersection(np.asarray(pairs.geometry),np.asarray(chips.geometry)[pairs.index_right])))
    centroids = geoms.geometry.to_crs(3035).centroid.to_crs(4326)
    location  = (pd.DataFrame({'id':geoms.id,'lon':centroids.x,'lat':centroids.y})
                 .merge(pairs.sort_values('overlap').drop_duplicates('id',keep='last')[['id','chip','split']],on='id',how='left'))

    table = (fields[['id','crop']].merge(location,on='id',validate='one_to_one').merge(features,on='id',validate='one_to_one')
             [['id','crop','chip','split','lon','lat']+names])
    table = drop_fields(table,table.split.notna(),'lie outside every chip, so have no split')
    table.to_parquet(PROCESSEDDIR/f"fields_{collection.split('/')[1]}.parquet",index=False)
    print(f'{collection} ends with {len(table):,} fields in {table.crop.nunique()} crops')
    return table

In [ ]:
pilot = build_table(PILOT)

In [ ]:
fig,ax = uplt.subplots(refwidth=5)
for split,color in SPLITCOLORS.items():
    subset = pilot[pilot.split==split]
    ax.scatter(subset.lon,subset.lat,s=0.3,color=color,label=f'{split} ({len(subset):,})',rasterized=True)
ax.format(xlabel='Longitude',ylabel='Latitude',title=f'{PILOT} fields by split',aspect=1/np.cos(np.radians(pilot.lat.mean())),grid=False)
ax.legend(loc='ul',ncols=1,markerscale=20,frame=False)
save_figure(fig,f"02_split_map_{PILOT.split('/')[1]}")

In [ ]:
cropcounts = pilot.crop.value_counts().sort_values()
fig,ax = uplt.subplots(refwidth=4,refheight=0.18*len(cropcounts))
ax.barh(cropcounts.index,cropcounts.values,color=SPLITCOLORS['train'],width=0.7)
ax.axvline(MINFIELDS,color='gray7',linestyle='--',linewidth=1,label=f'MINFIELDS = {MINFIELDS}')
ax.format(xscale='log',xformatter='{x:,.0f}',xlabel='Fields',ylabel='',title=f'{PILOT} fields per crop',ytickminor=False,ygrid=False,yticklabelsize=7)
ax.legend(loc='lr',frame=False)
save_figure(fig,f"02_crop_counts_{PILOT.split('/')[1]}")

In [ ]:
todo = [collection for collection in COLLECTIONS if not (PROCESSEDDIR/f"fields_{collection.split('/')[1]}.parquet").exists()]
print(f'{len(todo)} of {len(COLLECTIONS)} collections still need building')

for collection in todo:
    build_table(collection)

In [ ]:
summary = pd.DataFrame({collection.split('/')[1]:pd.read_parquet(PROCESSEDDIR/f"fields_{collection.split('/')[1]}.parquet",columns=['crop','split'])
                        .pipe(lambda table:{'crops':table.crop.nunique(),**table.split.value_counts().to_dict()}) for collection in COLLECTIONS}).T
summary = summary[['crops','train','val','test']].sort_values('train')
headers = {'crops':'Crops','train':'Train Fields','val':'Val Fields','test':'Test Fields'}

print(f'{summary[["train","val","test"]].values.sum():,} fields across {len(summary)} collections')
display(summary.rename(columns=headers).style.format('{:,}'))

In [ ]:
fig,ax = uplt.subplots(refwidth=4,refheight=0.25*len(summary))
ax.barh(summary.index,summary[list(SPLITCOLORS)].to_numpy(),stack=True,cycle=list(SPLITCOLORS.values()),labels=list(SPLITCOLORS),width=0.7)
ax.format(xlabel='Fields',ylabel='',xformatter='{x:,.0f}',title='Kept fields per collection',ytickminor=False,ygrid=False)
ax.legend(loc='b',ncols=3,frame=False)
save_figure(fig,'02_fields_per_collection')